# Synthetic Storm Experiment

## Step 1. Find "calm" year for each region

In [ ]:
import pandas as pd

# configuration

REGIONS = {
    "appalachia":    {"acy": r"ACY PATH", "sad": r"SAD PATH", "plant": "05-02"},
    "piedmont":      {"acy": r"ACY PATH", "sad": r"SAD PATH", "plant": "04-01"},
    "valley_ridge":  {"acy": r"ACY PATH", "sad": r"SAD PATH", "plant": "04-01"},
    "coastal_plain": {"acy": r"ACY PATH", "sad": r"SAD PATH", "plant": "04-01"},
}
HARVEST   = "10-02"
YEARS     = range(1990, 2000)
CROP_NAME = "CORN"
WINDOW    = 3

# 2-yr, 24-h Atlas-14 depth (mm)
ATLAS_2YR = {"appalachia": 62.0, "piedmont": 77.0, "valley_ridge": 71.0, "coastal_plain": 81.0}

SAD_KW = dict(skiprows=9, encoding="latin-1", sep=r"\s+")   
ACY_KW = dict(skiprows=8, encoding="latin-1", sep=r"\s+") 

COL_YEAR, COL_MON, COL_DAY = "Y", "M", "D"
COL_CROP_SAD, COL_CROP_ACY = "CPNM", "CPNM"
COL_HUI, COL_PRCP          = "HUI", "PRCP"      
COL_BIOM, COL_STD, COL_STL = "BIOM", "STD", "STL"
COL_YR_ACY, COL_YLD        = "YR", "YLDG"

# helper functions
def load_crop_yield(path):
    df = pd.read_csv(path, **ACY_KW)
    df = df[df[COL_CROP_ACY].astype(str).str.upper() == CROP_NAME]
    return df.set_index(COL_YR_ACY)[COL_YLD]


def load_daily(path):
    df = pd.read_csv(path, **SAD_KW)
    df = df[df[COL_CROP_SAD].astype(str).str.upper() == CROP_NAME].copy()
    df["date"] = pd.to_datetime(dict(year=df[COL_YEAR], month=df[COL_MON], day=df[COL_DAY]))
    keep = [COL_PRCP, COL_HUI, COL_BIOM, COL_STD, COL_STL]
    return df.set_index("date")[keep].sort_index()


def midseason_date(daily, year):
    yr = daily.loc[str(year)]
    hit = yr[yr[COL_HUI] >= 0.5]
    return hit.index[0] if len(hit) else None


def window_clean(daily, center, thresh):
    if center is None:
        return False
    lo, hi = center - pd.Timedelta(days=WINDOW), center + pd.Timedelta(days=WINDOW)
    nbrs = daily.loc[lo:hi].drop(index=center, errors="ignore")
    return bool((nbrs[COL_PRCP] <= thresh).all())


def crop_state(daily, date):
    """BIOM, STL, STD at a date; NaN if the date isn't present."""
    if date is None or date not in daily.index:
        return float("nan"), float("nan"), float("nan")
    b, std, stl = daily.loc[date, [COL_BIOM, COL_STD, COL_STL]]
    return round(b, 2), round(stl, 2), round(std, 2)


# Run

results = []
for region, cfg in REGIONS.items():
    yld   = load_crop_yield(cfg["acy"])
    daily = load_daily(cfg["sad"])
    thr   = ATLAS_2YR[region]
    med   = yld.reindex(YEARS).median()

    for year in YEARS:
        pre  = pd.Timestamp(f"{year}-{cfg['plant']}") - pd.Timedelta(days=3)
        post = pd.Timestamp(f"{year}-{HARVEST}")      + pd.Timedelta(days=3)
        mid  = midseason_date(daily, year)
        big  = (daily.loc[str(year), COL_PRCP] > thr).sum()

        checks = {"preplant": window_clean(daily, pre, thr),
                  "midseason": window_clean(daily, mid, thr),
                  "postharvest": window_clean(daily, post, thr)}

        pre_b,  pre_stl,  pre_std  = crop_state(daily, pre)
        mid_b,  mid_stl,  mid_std  = crop_state(daily, mid)
        post_b, post_stl, post_std = crop_state(daily, post)

        results.append({
            "region": region, "year": year,
            "corn_yield": round(yld.get(year, float("nan")), 2),
            "yld_dev": round(abs(yld.get(year, float("nan")) - med), 2),
            "annual_precip_mm": round(daily.loc[str(year), COL_PRCP].sum(), 0),
            "n_big_events": int(big),
            "pre_BIOM": pre_b,   "pre_STL": pre_stl,   "pre_STD": pre_std,
            "mid_BIOM": mid_b,   "mid_STL": mid_stl,   "mid_STD": mid_std,
            "post_BIOM": post_b, "post_STL": post_stl, "post_STD": post_std,
            **checks,
            "all_clean": all(checks.values()),
            "midseason_date": None if mid is None else mid.date(),
        })

out = pd.DataFrame(results).sort_values(
    ["region", "all_clean", "yld_dev", "n_big_events"],
    ascending=[True, False, True, True])

pd.set_option("display.width", 300, "display.max_columns", None)
for region in REGIONS:
    print(f"\n===== {region} =====")
    print(out[out.region == region].to_string(index=False))

## Step 2. Generate DLY & HLY files
Inject synthetic storms

In [ ]:
# configuration
from pathlib import Path

OUT_DIR = # where to output new weather files

# Per region: base HLY + base DLY + selected typical year.
REGIONS = {
    "appalachia":    {"hly": r"", "dly": r"", "year": 1993},
    "piedmont":      {"hly": r"", "dly": r"", "year": 1998},
    "valley_ridge":  {"hly": r"", "dly": r"", "year": 1998},
    "coastal_plain": {"hly": r"", "dly": r"", "year": 1993},
}

# Injection dates (month, day). VERIFY against your fresh step-1 output before running.
#   preplant = plant - 3 ;  midseason = HUI>=0.5 date ;  postharvest = kill(10/2) + 3 = 10/5
INJECT_DATES = {
    "appalachia":    {"preplant": (4, 29), "midseason": (8, 31), "postharvest": (10, 5)}, 
    "piedmont":      {"preplant": (4, 29), "midseason": (7, 23), "postharvest": (10, 5)},
    "valley_ridge":  {"preplant": (4, 29), "midseason": (7, 23),  "postharvest": (10, 5)},
    "coastal_plain": {"preplant": (4, 29), "midseason": (7, 21),  "postharvest": (10, 5)},
}

RETURN_PERIODS = [2, 5, 10, 25, 50, 100]
ATLAS_14 = {
    "appalachia":    {2: 62, 5: 81,  10: 96, 25: 116, 50: 131, 100: 147},
    "piedmont":      {2: 77, 5: 98,  10: 117, 25: 146, 50: 172, 100: 202},
    "valley_ridge":  {2: 71, 5: 88,  10: 103, 25: 124, 50: 141, 100: 160},
    "coastal_plain": {2: 82, 5: 107, 10: 128, 25: 161, 50: 189, 100: 221},
}

# SCS Type II 24-h cumulative ratio
SCS_TYPE_II = {1:0.0064, 2:0.0139, 3:0.0224, 4:0.0320, 5:0.0427, 6:0.0544,
               7:0.0690, 8:0.0878, 9:0.1108, 10:0.1454, 11:0.2057, 12:0.4763,
               13:0.7943, 14:0.8546, 15:0.8892, 16:0.9122, 17:0.9310, 18:0.9456,
               19:0.9573, 20:0.9680, 21:0.9776, 22:0.9861, 23:0.9936, 24:1.0000}

PRCP_LO, PRCP_HI = 32, 38


def read_keep_eol(path):
    """Read lines preserving original CRLF/LF, and report the line ending used."""
    lines = open(path, "r", encoding="latin-1", newline="").readlines()
    eol = "\r\n" if lines and lines[0].endswith("\r\n") else "\n"
    return lines, eol


def inject_hly(lines, eol, year, month, day, depth):
    out, n = [], 0
    for ln in lines:
        p = ln.split()
        if len(p) >= 4 and p[0].isdigit() and (int(p[0]), int(p[1]), int(p[2])) == (year, month, day):
            h = int(p[3]); n += 1
            out.append(f"{year:4d}{month:4d}{day:4d}{h:10d}{SCS_TYPE_II[h]*depth:10.3f}{eol}")
        else:
            out.append(ln)
    assert n == 24, f"HLY: overwrote {n} rows for {year}-{month:02d}-{day:02d} (expected 24)"
    return out


def inject_dly(lines, year, month, day, depth):
    out, n = [], 0
    for ln in lines:
        body = ln.rstrip("\r\n")
        try:
            ymd = (int(body[2:6]), int(body[6:10]), int(body[10:14]))
        except ValueError:
            out.append(ln); continue
        if ymd == (year, month, day):
            new = body[:PRCP_LO] + f"{depth:6.1f}" + body[PRCP_HI:] + ln[len(body):]
            assert len(new) == len(ln), "DLY: line length changed -- column misalignment"
            out.append(new); n += 1
        else:
            out.append(ln)
    assert n == 1, f"DLY: matched {n} rows for {year}-{month:02d}-{day:02d} (expected 1)"
    return out


Path(OUT_DIR).mkdir(exist_ok=True)
for region, cfg in REGIONS.items():
    hly_base, hly_eol = read_keep_eol(cfg["hly"])
    dly_base, _       = read_keep_eol(cfg["dly"])
    year = cfg["year"]
    for window, (mo, dy) in INJECT_DATES[region].items():
        for rp in RETURN_PERIODS:
            depth = ATLAS_14[region][rp]
            if depth is None:
                print(f"SKIP {region} {window} {rp}-yr: depth missing"); continue
            tag = f"{region}_{window}_{rp}yr"
            with open(Path(OUT_DIR) / f"{tag}.HLY", "w", encoding="latin-1", newline="") as f:
                f.writelines(inject_hly(hly_base, hly_eol, year, mo, dy, depth))
            with open(Path(OUT_DIR) / f"{tag}.dly", "w", encoding="latin-1", newline="") as f:
                f.writelines(inject_dly(dly_base, year, mo, dy, depth))
            print(f"wrote {tag}  ({depth} mm on {year}-{mo:02d}-{dy:02d})")

## Step 2.5: trim
Keep only 10 years of spinup (consistent with other scales)

In [ ]:
import os, glob, re

# configuration

INJECTED_DIR = # path to new weather files generated in previous step
SOURCE_ROOT  = # path to source scenarios
REGION_FOLDER = {"appalachia":"a-soy","piedmont":"p-soy","valley_ridge":"vr-soy","coastal_plain":"cp-soy"}
INJ_YEAR = {"appalachia":1993,"piedmont":1998,"valley_ridge":1998,"coastal_plain":1993}
SPINUP = 10                      # years of spin-up before the injection year
NBYR   = SPINUP + 1              # 11 total (spin-up + injection year)
REGIONS = list(REGION_FOLDER)

# helpers

def region_of(fname):
    return next(r for r in REGIONS if fname.startswith(r + "_"))

def trim(path, lo, hi, yr_slice):
    """Keep only lines whose year is in [lo, hi]. yr_slice: 'dly' or 'hly'."""
    out = []
    for ln in open(path, encoding="latin-1", newline=""):
        try:
            y = int(ln[2:6]) if yr_slice == "dly" else int(ln.split()[0])
        except (ValueError, IndexError):
            out.append(ln); continue          # keep unparseable (headers/blanks)
        if lo <= y <= hi:
            out.append(ln)
    open(path, "w", encoding="latin-1", newline="").writelines(out)

# trim injected weather
for f in glob.glob(os.path.join(INJECTED_DIR, "*.dly")) + glob.glob(os.path.join(INJECTED_DIR, "*.HLY")):
    region = region_of(os.path.basename(f))
    hi = INJ_YEAR[region]; lo = hi - SPINUP
    trim(f, lo, hi, "dly" if f.lower().endswith(".dly") else "hly")
print("trimmed injected weather")

# rewrite APEXCONT line 1: NBYR and IYR 
for region, rf in REGION_FOLDER.items():
    iyr = INJ_YEAR[region] - SPINUP
    for cont in glob.glob(os.path.join(SOURCE_ROOT, rf, "*", "APEXCONT.DAT")):
        lines = open(cont, encoding="latin-1").read().splitlines(keepends=True)
        vals = iter([str(NBYR), str(iyr)])
        lines[0] = re.sub(r"\d+", lambda m: next(vals), lines[0], count=2)
        open(cont, "w", encoding="latin-1").writelines(lines)
        print(f"  {region}: NBYR={NBYR} IYR={iyr}  <- {cont}")

## Step 3. Create scenarios + run

In [ ]:
import os, glob, shutil, time, subprocess, csv
from pathlib import Path

# configuration

INJECTED_DIR =  # path to new weather files generated in step 2
SOURCE_ROOT  = # path to source files
OUTPUT_ROOT  = # path to output directory
EXE_NAME     = "apex-mks.exe"
MANIFEST     = # path and filename for harness csv

MAX_PARALLEL = 4
TIMEOUT_S    = 120          # kill a run that hangs past this
PRUNE        = True         # after each run, keep only .SAD/.ACY
RESUME       = False         # skip tasks whose .SAD already exists
VALIDATED    = True        # False = run ONE validation task, then stop

REGION_FOLDER = {"appalachia": "a-soy", "piedmont": "p-soy",
                 "valley_ridge": "vr-soy", "coastal_plain": "cp-soy"}
DELTA_T = {"0": 0.0, "45": 2.03, "85": 2.70}   # label -> offset (applied to whole .dly)
REGIONS = ["appalachia", "piedmont", "valley_ridge", "coastal_plain"]

# helpers 

def robust_delete(path, tries=6, wait=0.5):
    """Delete a file/dir, retrying through transient Windows locks (exe just exited / AV scan)."""
    for i in range(tries):
        try:
            if os.path.isdir(path):
                shutil.rmtree(path)
            elif os.path.isfile(path):
                os.remove(path)
            return True
        except PermissionError:
            time.sleep(wait * (i + 1))      # back off: 0.5s, 1s, 1.5s, ...
    print(f"  WARN: could not delete (locked): {path}")
    return False

def shift_temps(lines, offset):
    """Add offset to TMAX (cols 21-26) and TMIN (cols 27-32) on every row. PRCP untouched."""
    if offset == 0:
        return lines
    out = []
    for ln in lines:
        body = ln.rstrip("\r\n")
        try:
            tmax = float(body[20:26]) + offset
            tmin = float(body[26:32]) + offset
        except ValueError:
            out.append(ln); continue
        new = body[:20] + f"{tmax:6.1f}" + f"{tmin:6.1f}" + body[32:] + ln[len(body):]
        if len(new) != len(ln):
            raise ValueError("dly width changed after temp shift -- column misalignment")
        out.append(new)
    return out


def parse_injected(fname):
    """'valley_ridge_preplant_2yr.dly' -> (region, window, rp_int)."""
    stem = fname[:-4]
    for region in REGIONS:
        if stem.startswith(region + "_"):
            window, rpyr = stem[len(region) + 1:].split("_")
            return region, window, int(rpyr.replace("yr", ""))
    raise ValueError(f"cannot parse {fname}")


def build_task_list():
    tasks = []
    for f in sorted(glob.glob(os.path.join(INJECTED_DIR, "*.dly"))):
        region, window, rp = parse_injected(os.path.basename(f))
        rfolder = REGION_FOLDER[region]
        run_dirs = [d for d in glob.glob(os.path.join(SOURCE_ROOT, rfolder, "*")) if os.path.isdir(d)]
        for label, offset in DELTA_T.items():
            scen = f"{region}_{window}_{rp}yr_dT{label}"
            for src_run in run_dirs:
                dest = os.path.join(OUTPUT_ROOT, rfolder, scen, os.path.basename(src_run))
                tasks.append(dict(region=region, window=window, rp=rp, label=label,
                                  offset=offset, src_run=src_run, dest=dest,
                                  inj_stem=f"{region}_{window}_{rp}yr"))
    return tasks


def build(task):
    """Copy the run folder, swap in the injected _2su weather (with ΔT applied to .dly)."""
    dest = task["dest"]
    if RESUME and glob.glob(os.path.join(dest, "*.SAD")):
        return None                                   # already done
    if os.path.exists(dest):
        robust_delete(dest)
    shutil.copytree(task["src_run"], dest)

    dly_target = glob.glob(os.path.join(dest, "*_2su.dly"))
    hly_target = glob.glob(os.path.join(dest, "*_2su.hly"))
    assert len(dly_target) == 1 and len(hly_target) == 1, f"expected one _2su pair in {dest}"

    # HLY: copy injected storm verbatim
    shutil.copyfile(os.path.join(INJECTED_DIR, task["inj_stem"] + ".HLY"), hly_target[0])
    # DLY: injected storm + (optional) temperature offset on all rows
    inj = open(os.path.join(INJECTED_DIR, task["inj_stem"] + ".dly"),
               encoding="latin-1", newline="").readlines()
    with open(dly_target[0], "w", encoding="latin-1", newline="") as fh:
        fh.writelines(shift_temps(inj, task["offset"]))
    return dest


def prune(dest):
    time.sleep(0.3)                          # let the OS release the just-exited exe handle
    for p in list(Path(dest).iterdir()):
        if p.is_file() and p.suffix.upper() not in (".SAD", ".ACY", ".DHY", ".DLY", ".HLY", ".MSA", ".DCN"):
            robust_delete(str(p))


def reap(running, manifest):
    """Poll running jobs; collect finished/timed-out, prune, return survivors."""
    still = []
    for proc, dest, t0, task in running:
        rc = proc.poll()
        if rc is None and time.time() - t0 > TIMEOUT_S:
            proc.kill(); rc = "TIMEOUT"
        if rc is None:
            still.append((proc, dest, t0, task))
        else:
            ok = (rc == 0)
            if ok and PRUNE:
                try:
                    prune(dest)
                except Exception as e:
                    print(f"  WARN prune failed {dest}: {e}")
            manifest.append({**{k: task[k] for k in ("region", "window", "rp", "label")},
                "dest": dest, "rc": rc})

            print(f"  done rc={rc}: {os.path.relpath(dest, OUTPUT_ROOT)}")
    if len(still) == len(running):
        time.sleep(0.2)
    return still


# Run
tasks = build_task_list()
print(f"{len(tasks)} total run-tasks "
      f"({len(set((t['region'],t['window'],t['rp'],t['label']) for t in tasks))} scenarios x 7 runs)")

# validation gate: run ONE warm-region base run first
val = next((t for t in tasks if t["region"] == "piedmont" and t["window"] == "preplant"
            and t["rp"] == 2 and t["label"] == "0" and t["dest"].endswith("base-1990-2000")), tasks[0])
if not VALIDATED:
    d = build(val) or val["dest"]
    subprocess.run([os.path.join(d, EXE_NAME)], cwd=d,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    sad = glob.glob(os.path.join(d, "*.SAD"))
    print(f"\nVALIDATION run complete. Inspect: {sad[0] if sad else '(no SAD produced!)'}")
    print("Check storm-day PRCP ~= depth and Q > 0, then set VALIDATED=True and rerun.")
else:
    manifest, running = [], []
    for task in tasks:
        dest = build(task)
        if dest is None:
            continue                                  # resumed/skip
        proc = subprocess.Popen([os.path.join(dest, EXE_NAME)], cwd=dest,
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        running.append((proc, dest, time.time(), task))
        while len(running) >= MAX_PARALLEL:
            running = reap(running, manifest)
    while running:
        running = reap(running, manifest)

    with open(MANIFEST, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["region", "window", "rp", "label", "dest", "rc"])
        w.writeheader(); w.writerows(manifest)
    fails = [m for m in manifest if m["rc"] != 0]
    print(f"\nfinished {len(manifest)} runs, {len(fails)} non-zero. Manifest: {MANIFEST}")

## Step 4. Extract Data

### by date

In [ ]:

import os, glob
import pandas as pd

# Configuration

RUNS_ROOT = # path to experiment scenario files
OUT_CSV   = # path to output csv + filename

REGION_FOLDER = {"appalachia": "a-soy", "piedmont": "p-soy",
                 "valley_ridge": "vr-soy", "coastal_plain": "cp-soy"}
REGION_YEAR   = {"appalachia": 1993, "piedmont": 1998, "valley_ridge": 1998, "coastal_plain": 1993} # soy
# {"appalachia": 1994, "piedmont": 1993, "valley_ridge": 1999, "coastal_plain": 1995} # corn

INJECT_DATES = {
    # corn
    # "appalachia":    {"preplant": (4, 29), "midseason": (8, 4), "postharvest": (10, 5)},
    # "piedmont":      {"preplant": (3, 29), "midseason": (6, 27), "postharvest": (10, 5)},
    # "valley_ridge":  {"preplant": (3, 29), "midseason": (7, 6),  "postharvest": (10, 5)},
    # "coastal_plain": {"preplant": (3, 29), "midseason": (7, 7),  "postharvest": (10, 5)},
    # soy
    "appalachia":    {"preplant": (4, 29), "midseason": (8, 31), "postharvest": (10, 5)},                       
    "piedmont":      {"preplant": (4, 29), "midseason": (7, 23), "postharvest": (10, 5)},
    "valley_ridge":  {"preplant": (4, 29), "midseason": (7, 23),  "postharvest": (10, 5)},
    "coastal_plain": {"preplant": (4, 29), "midseason": (7, 21),  "postharvest": (10, 5)},
}
DELTA_T = {"0": 0.0, "45": 2.03, "85": 2.70}
REGIONS = list(REGION_FOLDER)
SAD_KW = dict(skiprows=9, encoding="latin-1", sep=r"\s+")
ACY_KW = dict(skiprows=8, encoding="latin-1", sep=r"\s+")
EPS = 1e-6

MSA_COLS = ['SA','ID','YR','YRNUM','VAR','JAN','FEB','MAR','APR','MAY','JUN',
            'JUL','AUG','SEP','OCT','NOV','DEC','ANNUAL','VAR2']
MONTHS = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

DHY_COLS = ['ISA','NBSA','Y','M','D','CN','SCI','RFV','STMP2','SML','Q','SSF',
            'QRF','RSSF','WYLD','QRB','TC','DUR','ALTC','AL5','REP','RZSW','GWST']


import pandas as pd, glob, os

# helpers

def rfv_on_injection(run_dir, inj_year, inj_month, inj_day):
    """Return RFV (mm APEX actually used) on the injection date, or None if no DHY / date missing."""
    dhy = glob.glob(os.path.join(run_dir, "*.DHY"))
    if not dhy:
        return None
    df = pd.read_csv(dhy[0], skiprows=10, sep=r"\s+", names=DHY_COLS, engine="python")
    r = df[(df.Y == inj_year) & (df.M == inj_month) & (df.D == inj_day)]
    return float(r.RFV.iloc[0]) if len(r) else None

def soil_state(run_folder, year, mo):
    """ZNH3/ZNO3/ZPML from the month BEFORE injection (avoids storm-induced depletion)."""
    out = {"ZNH3": float("nan"), "ZNO3": float("nan"), "ZPML": float("nan")}
    msa_f = glob.glob(os.path.join(run_folder, "*.MSA"))
    if not msa_f:
        return out
    msa = pd.read_csv(msa_f[0], skiprows=10, sep=r"\s+", names=MSA_COLS,
                       encoding="latin-1", engine="python")
    prior_mo, prior_yr = (mo - 1, year) if mo > 1 else (12, year - 1)
    col = MONTHS[prior_mo - 1]
    for var in out:
        row = msa[(msa.YR == prior_yr) & (msa.VAR == var)]
        if len(row):
            out[var] = float(row.iloc[0][col])
    return out

def parse_scen(name):
    region, rest = next((r, name[len(r) + 1:]) for r in REGIONS if name.startswith(r + "_"))
    left, label = rest.rsplit("_dT", 1)
    window, rpyr = left.split("_")
    return region, window, int(rpyr.replace("yr", "")), label


def bmp_of(run_folder, region_folder):
    return os.path.basename(run_folder).replace(region_folder + "-", "").replace("-1990-2000", "")

def day_row(sad, y, m, d):
    r = sad[(sad.Y == y) & (sad.M == m) & (sad.D == d)]
    return r.iloc[0] if len(r) else None

EVENT_DAYS = 2   # D and D+1 -- event fully closed by D+2 (verified: Q=0 at D+2)

def event_rows(sad, year, mo, dy):
    """Return the EVENT_DAYS rows starting at the injection day."""
    start = pd.Timestamp(year, mo, dy)
    dates = [(start + pd.Timedelta(days=i)) for i in range(EVENT_DAYS)]
    keys  = [(d.year, d.month, d.day) for d in dates]
    rows  = [day_row(sad, *k) for k in keys]
    return [r for r in rows if r is not None]


def read_run(run_folder, year, mo, dy):
    sad_f = glob.glob(os.path.join(run_folder, "*.SAD"))
    acy_f = glob.glob(os.path.join(run_folder, "*.ACY"))
    if not sad_f:
        return None
    sad = pd.read_csv(sad_f[0], **SAD_KW)
    rfv = rfv_on_injection(run_folder, year, mo, dy)
    ev = event_rows(sad, year, mo, dy)
    if not ev:
        return None
    ev = pd.DataFrame(ev)

    # check-day: the day after the event window should be ~0 
    chk = day_row(sad, *(pd.Timestamp(year, mo, dy) + pd.Timedelta(days=EVENT_DAYS)).timetuple()[:3])

    yldg = float("nan")
    if acy_f:
        acy = pd.read_csv(acy_f[0], **ACY_KW)
        cy = acy[(acy.YR == year) & (acy.CPNM.astype(str).str.upper() == "SOY")]
        if len(cy):
            yldg = cy.YLDG.iloc[0]

    # fluxes summed over the event window, state taken at the storm day (D)
    d0 = ev.iloc[0]
    return dict(
        PRCP=ev.PRCP.sum(), Q=ev.Q.sum(), WYLD=ev.WYLD.sum(), MUSL=ev.MUSL.sum(),
        YN=ev.YN.sum(), QN=ev.QN.sum(), YP=ev.YP.sum(), QP=ev.QP.sum(),
        TN=(ev.YN + ev.QN).sum(), TP=(ev.YP + ev.QP).sum(),
        BIOM=d0.BIOM, STL=d0.STL, STD=d0.STD, HUI=d0.HUI,   # state on D, not summed
        YLDG=yldg, Q_check=(float("nan") if chk is None else chk.Q),
        RFV=rfv, 
        **soil_state(run_folder, year, mo)
    )


def re_val(ref, bmp):
    return float("nan") if (pd.isna(ref) or ref < EPS) else (ref - bmp) / ref


# run

rows = []
for region in REGIONS:
    rf = REGION_FOLDER[region]
    year = REGION_YEAR[region]
    for scen_dir in glob.glob(os.path.join(RUNS_ROOT, rf, "*")):
        if not os.path.isdir(scen_dir):
            continue
        _, window, rp, label = parse_scen(os.path.basename(scen_dir))
        if window not in INJECT_DATES[region]:
            print(f"  SKIP invalid window: {os.path.basename(scen_dir)}")
            continue
        mo, dy = INJECT_DATES[region][window]

        runs = {}
        for run_folder in glob.glob(os.path.join(scen_dir, "*")):
            if os.path.isdir(run_folder):
                data = read_run(run_folder, year, mo, dy)
                if data:
                    runs[bmp_of(run_folder, rf)] = data

        base, mabase = runs.get("base"), runs.get("ma-base")
        for bmp, d in runs.items():
            ref = mabase if bmp == "ma" else base
            undef = ref is None or ref["TN"] < EPS
            rows.append(dict(
                region=region, window=window, return_period=rp, dT=DELTA_T[label],
                inj_date=f"{year}-{mo:02d}-{dy:02d}", bmp=bmp,
                **{k: round(d[k], 4) for k in d},
                RE_TN=None if (bmp in ("base", "ma-base") or ref is None) else round(re_val(ref["TN"], d["TN"]), 4),
                RE_TP=None if (bmp in ("base", "ma-base") or ref is None) else round(re_val(ref["TP"], d["TP"]), 4),
                RE_sed=None if (bmp in ("base", "ma-base") or ref is None) else round(re_val(ref["MUSL"], d["MUSL"]), 4),
                base_TN=None if ref is None else round(ref["TN"], 4),
                base_Q=None if ref is None else round(ref["Q"], 4),
                undefined=bool(undef),
            ))

out = pd.DataFrame(rows).sort_values(["region", "window", "return_period", "dT", "bmp"])
out.to_csv(OUT_CSV, index=False)
print(f"wrote {len(out)} rows -> {OUT_CSV}")
print(f"undefined (zero-baseline) cells: {int(out.undefined.sum())} / {len(out)}")